<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/12-Improve_Query.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install Packages and Setup Variables


In [ ]:
!pip install -q llama-index==0.14.0 openai==1.82.0 llama-index-embeddings-huggingface==0.7.0 \
                llama-index-embeddings-cohere==0.8.0 cohere==5.15.0 chromadb==1.0.12 jedi==0.19.2 \
                llama-index-vector-stores-chroma==0.5.5 llama-index-llms-google-genai==0.3.0 \
                llama-index-llms-openai==0.5.6 ipywidgets==8.1.5

In [ ]:
import os

# Set the following API Keys in the Python environment. Will be used later.
# os.environ["OPENAI_API_KEY"] = "<YOUR-OPENAI-API-KEY>"
# os.environ["GOOGLE_API_KEY"] = "<YOUR-GOOGLE-API-KEY>"

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    os.environ["GOOGLE_API_KEY"] = userdata.get('Google_api_key')
except ImportError:
    os.environ.setdefault("OPENAI_API_KEY", os.environ.get("OPENAI_API_KEY", ""))
    os.environ.setdefault("GOOGLE_API_KEY", os.environ.get("GOOGLE_API_KEY", ""))

In [ ]:
# Allows running asyncio in environments with an existing event loop, like Jupyter notebooks.
import nest_asyncio

nest_asyncio.apply()

# Load a Model


In [ ]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings

Settings.llm = OpenAI(model="gpt-4o-mini")
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

# Load Indexes


In [ ]:
import shutil
from huggingface_hub import hf_hub_download

# Use local vectorstore if available, otherwise download from HuggingFace Hub
_local_vs = "/Users/jai/Documents/code-repo/ai-engineering-v2-notebooks-claude-skill/ai-engineering-v2-notebooks-claude-skill/notebooks/vectorstore.zip"
if os.path.exists(_local_vs):
    shutil.copy(_local_vs, "vectorstore.zip")
    vectorstore = "vectorstore.zip"
else:
    vectorstore = hf_hub_download(repo_id="jaiganesan/ai_tutor_knowledge", filename="vectorstore.zip", repo_type="dataset", local_dir=".")

In [ ]:
import zipfile
if not os.path.exists("ai_tutor_knowledge"):
    with zipfile.ZipFile("vectorstore.zip", 'r') as zf:
        zf.extractall(".")

In [ ]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import VectorStoreIndex

# Create the vector index
db = chromadb.PersistentClient(path="./ai_tutor_knowledge")
chroma_collection = db.get_or_create_collection("ai_tutor_knowledge")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
vector_index = VectorStoreIndex.from_vector_store(vector_store)

# Multi-Step Query Engine


## GPT-4o-mini


In [ ]:
from llama_index.core.indices.query.query_transform.base import (
    StepDecomposeQueryTransform,
)

step_decompose_transform_gpt5 = StepDecomposeQueryTransform(verbose=True, llm=Settings.llm)

In [ ]:
from llama_index.core.query_engine.multistep_query_engine import MultiStepQueryEngine

#Default query engine
query_engine_gpt5_mini = vector_index.as_query_engine()

# Multi Step Query Engine
multi_step_query_engine = MultiStepQueryEngine(
    query_engine = query_engine_gpt5_mini,
    query_transform = step_decompose_transform_gpt5,
    index_summary = "Used to answer questions about RAG, Machine Learning, Deep Learning, and Generative AI. Note: Don't repeat the same question.",
)

# Query Dataset

## Default

In [ ]:
# Default query engine
query_engine = vector_index.as_query_engine()
res = query_engine.query("Write about Llama 3.1 Model, BERT and PEFT methods")
print(res.response)

In [ ]:
for src in res.source_nodes:
    print("Node ID\t", src.node_id)
    print("Title\t", src.metadata["title"])
    print("Text\t", src.text)
    print("Score\t", src.score)
    print("-_" * 20)

## GPT-4o-mini Multi-Step


In [ ]:
response = multi_step_query_engine.query("Write about Llama 3.1 Model, BERT and PEFT methods")
print(response.response)

In [ ]:
for query, response in response.metadata['sub_qa']:
    print(f"**{query}**\n{response}\n")

In [ ]:
for src in response.source_nodes:
    print("Node ID\t", src.node_id)
    print("Text\t", src.text)
    print("Score\t", src.score)
    print("-_" * 20)

# Test gemini-2.5-flash Multi-Step


In [ ]:
from llama_index.core.indices.query.query_transform.base import StepDecomposeQueryTransform
from llama_index.core.query_engine.multistep_query_engine import MultiStepQueryEngine
from llama_index.llms.google_genai import GoogleGenAI

llm_gemini = GoogleGenAI(model="gemini-2.5-flash")

step_decompose_transform = StepDecomposeQueryTransform(llm=llm_gemini, verbose=True)

query_engine_gemini = vector_index.as_query_engine(llm=llm_gemini)

query_engine_gemini = MultiStepQueryEngine(
    query_engine=query_engine_gemini,
    query_transform=step_decompose_transform,
    index_summary="Used to answer the Questions about RAG, Machine Learning, Deep Learning,LLMs and Generative AI",
)

In [ ]:
response_gemini = query_engine_gemini.query("Write about Llama 3.1 Model, BERT and PEFT")

In [ ]:
response_gemini.response

## Test Retriever on Multistep


In [ ]:
# import llama_index
# from llama_index.core.indices.query.schema import QueryBundle

# t = QueryBundle("How Retrieval Augmented Generation (RAG) work?")
# query_engine_gemini.retrieve(t)

## Subquestion Query Engine

In [ ]:
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.query_engine import SubQuestionQueryEngine
from llama_index.core.question_gen.llm_generators import LLMQuestionGenerator
from llama_index.llms.openai import OpenAI

llm = OpenAI(model="gpt-4o-mini")

question_gen = LLMQuestionGenerator.from_defaults(llm=llm)

query_engine = vector_index.as_query_engine()

query_engine_tools = [
    QueryEngineTool(
        query_engine=query_engine,
        metadata=ToolMetadata(
            name="LlamaIndex",
            description="Used to answer the Questions about RAG, Machine Learning, Deep Learning, and Generative AI. Note: Don't repeat the Same question",
        ),
    ),
]

sub_question_engine = SubQuestionQueryEngine.from_defaults(
    query_engine_tools=query_engine_tools,
    question_gen=question_gen,
    use_async=True,
)

response = sub_question_engine.query("Write about Llama 3.1 Model, BERT and PEFT")

In [ ]:
response.response

# HyDE Transform


In [ ]:
query_engine = vector_index.as_query_engine()

In [ ]:
from llama_index.core.indices.query.query_transform import HyDEQueryTransform
from llama_index.core.query_engine.transform_query_engine import TransformQueryEngine

hyde = HyDEQueryTransform(include_original=True)
hyde_query_engine = TransformQueryEngine(query_engine, hyde)

In [ ]:
response = hyde_query_engine.query("Write about Llama 3.1 Model, BERT and PEFT")

In [ ]:
response.response

In [ ]:
for src in response.source_nodes:
    print("Node ID\t", src.node_id)
    print("Text\t", src.text)
    print("Score\t", src.score)
    print("-_" * 20)

In [ ]:
query_bundle = hyde("Write about Llama 3.1 Model, BERT and PEFT")

In [ ]:
hyde_doc = query_bundle.embedding_strs[0]

In [ ]:
hyde_doc

## Optional: Interactive Query Strategy Comparison

Use the widget below to compare Default, Multi-Step, HyDE, and Sub-Question query strategies interactively.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, Markdown

    strategy_dropdown = widgets.Dropdown(
        options=["Default", "Multi-Step (Gemini)", "HyDE"],
        description="Strategy:",
        style={"description_width": "initial"},
    )
    query_input = widgets.Text(
        value="Write about Llama 3.1 Model and BERT",
        description="Query:",
        layout=widgets.Layout(width="60%"),
        style={"description_width": "initial"},
    )
    run_button = widgets.Button(description="Run Query", button_style="primary")
    output_area = widgets.Output()

    def on_run_clicked(b):
        output_area.clear_output()
        with output_area:
            q = query_input.value.strip()
            strategy = strategy_dropdown.value
            print(f"Running strategy: {strategy}")
            print(f"Query: {q}\n")
            if strategy == "Default":
                r = vector_index.as_query_engine().query(q)
            elif strategy == "Multi-Step (Gemini)":
                r = query_engine_gemini.query(q)
            elif strategy == "HyDE":
                r = hyde_query_engine.query(q)
            print(r.response)

    run_button.on_click(on_run_clicked)
    display(widgets.VBox([query_input, strategy_dropdown, run_button, output_area]))
except ImportError:
    print("ipywidgets not installed. Run: pip install ipywidgets")